In [1]:
import torch
import numpy as np
from datasets import load_dataset,DatasetDict, Dataset
import os
import time
import math
import tiktoken
import sys
import numpy as np
from tqdm.auto import tqdm
from transformers import GPT2Tokenizer
import json
import argparse

from contextlib import nullcontext

In [2]:
RELOAD_DATA_WIKI = False
RELOAD_DATA_TINY = True

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
original_dir = os.getcwd()
work_dir = "/content/drive/MyDrive/DiffusionLLMs/"

In [5]:
sys.path.append(work_dir)
os.chdir(work_dir)

In [6]:
from sinusoidal_position import SinusoidalPosition
from ssharma_gpt import GPTModel
import matplotlib.pyplot as plt
import pickle

In [7]:
os.chdir(original_dir)

In [10]:
if RELOAD_DATA_WIKI:
    ds_wiki = load_dataset(
      "Salesforce/wikitext",
      "wikitext-103-v1"
      )["train"]

    lines = []
    for entry in ds_wiki:
      text = entry["text"]
      text = text.replace("\n", " ") # remove newline formattin
      text = " ".join(text.split()) # remove sequences of whitespace
      lines.append(text+"\n")

    os.makedirs("/content/drive/MyDrive/DiffusionLLMs/", exist_ok=True)

    with open("/content/drive/MyDrive/DiffusionLLMs/data.txt", "w") as f:
        f.writelines(lines)
    f.close

In [11]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

In [12]:
def tokenize_function(examples):
    outputs = tokenizer(
        examples["text"],
        truncation=True,
        max_length=512
    )
    return {
        "ids": outputs["input_ids"],
        "len": [len(x) for x in outputs["input_ids"]]
    }

In [13]:
def create_tokenized_binary_file(tokenized_input,dataset_name):
  os.makedirs(work_dir, exist_ok=True)

  if isinstance(tokenized_input, Dataset):
        # Default split name if a single Dataset is passed
        items_to_process = {"train": tokenized_input}
  elif isinstance(tokenized_input, DatasetDict):
        items_to_process = tokenized_input.items()
  elif isinstance(tokenized_input, dict):
        items_to_process = tokenized_input.items()
  else:
        raise TypeError(f"Unsupported dataset type: {type(tokenized_input)}")

  for split, dset in tokenized_input.items():
        filename = os.path.join(work_dir, f"{dataset_name}_{split}.bin")

        # Check existence per-split file
        if os.path.exists(filename):
            print(f"Skipping {filename} (already exists. overwriting)")
        arr_len = np.sum(dset['len'], dtype=np.uint64)

        dtype = np.uint16 # (can do since enc.max_token_value == 50256 is < 2**16)
        arr = np.memmap(filename, dtype=dtype, mode='w+', shape=(arr_len,))
        total_batches = 1024

        idx = 0
        for batch_idx in tqdm(range(total_batches), desc=f'writing {filename}'):
            # Batch together samples for faster write
            batch = dset.shard(num_shards=total_batches, index=batch_idx, contiguous=True).with_format('numpy')
            arr_batch = np.concatenate(batch['ids'])
            # Write into mmap
            arr[idx : idx + len(arr_batch)] = arr_batch
            idx += len(arr_batch)
        arr.flush()

In [14]:
if RELOAD_DATA_WIKI:
  # First split: 80% train, 20% temp (which will become val + test)
  ds_split1 = ds_wiki.train_test_split(test_size=0.2, seed=42)

  # Second split: Split the 20% temp evenly into 10% val and 10% test
  ds_split2 = ds_split1['test'].train_test_split(test_size=0.5, seed=42)

  ds_wiki_3splits = DatasetDict({
    'train': ds_split1['train'],
    'val': ds_split2['train'],
    'test': ds_split2['test']
  })

In [15]:
if RELOAD_DATA_WIKI:
  tokenized_ds_wiki = ds_wiki_3splits.map(
      tokenize_function,
      batched=True,
      num_proc=8,
      remove_columns=["text"]
  )

In [17]:
if RELOAD_DATA_WIKI:
  create_tokenized_binary_file(tokenized_ds_wiki, "wiki")

In [ ]:
if RELOAD_DATA_TINY:
    ds_tiny = load_dataset("roneneldan/TinyStories")

    # Extract train, validation, and test splits
    # TinyStories has 'train' and 'validation'; we can split validation into val and test
    val_test_split = ds_tiny['validation'].train_test_split(test_size=0.5, seed=42)
    ds_tiny_3splits = DatasetDict({
        'train': ds_tiny['train'],
        'val': val_test_split['train'],
        'test': val_test_split['test']
    })

    # Tokenize splits
    tokenized_ds_tiny = ds_tiny_3splits.map(
        tokenize_function,
        batched=True,
        num_proc=8,
        remove_columns=["text"]
    )

    # Save mmap binary files as 'tinystories_train.bin', etc.
    create_tokenized_binary_file(tokenized_ds_tiny, "tinystories")

README.md:   0%|          | 0.00/1.06k [00:00<?, ?B/s]

data/train-00000-of-00004-2d5a1467fff108(…): reconstructing file:   0%|          |  0.00B /  249MB            

data/train-00000-of-00004-2d5a1467fff108(…): downloading bytes:           |  0.00B            

data/train-00001-of-00004-5852b56a2bd28f(…): reconstructing file:   0%|          |  0.00B /  248MB            

data/train-00001-of-00004-5852b56a2bd28f(…): downloading bytes:           |  0.00B            

...

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

Map (num_proc=8):   0%|          | 0/2119719 [00:00<?, ? examples/s]

Map (num_proc=8):   0%|          | 0/10995 [00:00<?, ? examples/s]

Map (num_proc=8):   0%|          | 0/10995 [00:00<?, ? examples/s]

writing /content/drive/MyDrive/DiffusionLLMs/tinystories_train.bin:   0%|          | 0/1024 [00:00<?, ?it/s]

## GPT Model
This model can be configured either as a Autoregressive or as a Diffusion based model

In [ ]:
def get_config():
    config_filename = os.path.join(work_dir,"llm_config.json")
    try:
        with open(config_filename, 'r') as config_file:
            config_data = json.load(config_file)
            return config_data

    except FileNotFoundError:
        print("Error: llm_config.json not found.")
        return None

In [ ]:
def get_positional_embedding(config_data):
    positional_embedding = config_data['llm_config']['positional_embedding']
    print(positional_embedding)
    return positional_embedding

In [ ]:
def get_device_and_precision():
    """Selects target hardware device, precision dtype, autocast context, and gradient scaler."""
    device = "cuda" if torch.cuda.is_available() else "cpu"
    device_type = "cuda" if "cuda" in device else "cpu"

    dtype = 'bfloat16' if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else 'float16'
    ptdtype = {'float32': torch.float32, 'bfloat16': torch.bfloat16, 'float16': torch.float16}[dtype]

    ctx = nullcontext() if device_type == 'cpu' else torch.amp.autocast(device_type=device_type, dtype=ptdtype)
    if device_type == 'cpu':
        scaler = torch.amp.GradScaler(enabled=False)
    else:
      scaler = torch.amp.GradScaler(enabled=(dtype == 'float16'))

    print(f"Using device: {device} | Precision: {dtype}")
    return device, dtype, ctx, scaler

In [ ]:
def init_model(config_data):
    llm_type = config_data['llm_config'].get('LLM_type') or config_data['llm_config'].get('llm_type')
    print("LLM Type ",llm_type)
    config_data["llm_config"]["vocab_size"] = 50257 # Hardcoded
    positional_embedding=get_positional_embedding(config_data)
    d_model = config_data['llm_config']['d_model']
    max_seq_len=config_data['llm_config']['max_seq_len']
    n_heads = config_data['llm_config']['n_heads']
    num_layers=config_data['llm_config']['num_layers']
    vocab_size=config_data['llm_config']['vocab_size']
    attention_type=config_data['llm_config']['attention']
    theta=config_data['llm_config']['theta']
    print("Theta for Rotary Embedings ",theta)
    ffn_hidden_dim=config_data['llm_config']['ffn_hidden_dim']
    ffn_hidden_dim=int(ffn_hidden_dim)
    print("FFN Hidden Dim ",ffn_hidden_dim)
    if llm_type=="Autoregressive":
      activation="relu"
    else:
      activation="gelu"
    print("Activation ",activation)
    if llm_type=="Diffusion":
      diffusion_steps=config_data['llm_config'].get('diffusion_steps')
      print("Diffusion Steps ",diffusion_steps)
      config_data['llm_config']['diffusion_steps']=diffusion_steps
      model = GPTModel(d_model, n_heads, num_layers, vocab_size, max_seq_len,\
                     positional_embedding,theta,attention_type,ffn_hidden_dim,llm_type,activation,diffusion_steps)
    else:
      model = GPTModel(d_model, n_heads, num_layers, vocab_size, max_seq_len,\
                     positional_embedding,theta,attention_type,ffn_hidden_dim,llm_type,activation)
    param_count = sum(p.numel() for p in model.parameters())
    print("Model has", param_count, "parameters.")
    return model

In [ ]:
def get_batch(split, dataset_dir, dataset_name, block_size, batch_size, device):
    """Fetches a batch of input (x) and target (y) sequences from binary files."""
    filename = os.path.join(dataset_dir, f"{dataset_name}_{split}.bin")
    if not os.path.exists(filename):
        raise FileNotFoundError(f"Binary dataset not found at {filename}")

    data = np.memmap(filename, dtype=np.uint16, mode='r')

    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([torch.from_numpy((data[i:i+block_size]).astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy((data[i+1:i+1+block_size]).astype(np.int64)) for i in ix])

    return x.to(device), y.to(device)

In [ ]:
@torch.no_grad()
def estimate_loss(model, eval_iters, dataset_dir, dataset_name, block_size, batch_size, device, ctx):
    """Evaluates loss across train and val splits without calculating gradients."""
    out = {}
    model.eval()

    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X_eval, Y_eval = get_batch(split, dataset_dir, dataset_name, block_size, batch_size, device)
            with ctx:
                logits, loss = model(X_eval, targets=Y_eval) if hasattr(model, 'targets') else (model(X_eval), None)
                if loss is None:
                    loss = torch.nn.functional.cross_entropy(logits.view(-1, logits.size(-1)), Y_eval.view(-1))
            losses[k] = loss.item()
        out[split] = losses.mean()

    model.train()
    return out

In [ ]:
def plot_loss_curves(train_losses, val_losses, eval_interval=500, dataset_name="wiki"):

    steps = [i * eval_interval for i in range(len(val_losses))]

    plt.figure(figsize=(10, 6))
    plt.plot(steps, train_losses, label="Train Loss", color="#1f77b4", linewidth=2)
    plt.plot(steps, val_losses, label="Validation Loss", color="#ff7f0e", linewidth=2, linestyle="--")

    plt.title("GPT Training & Validation Loss", fontsize=14, fontweight="bold")
    plt.xlabel("Iteration Step", fontsize=12)
    plt.ylabel("Cross-Entropy Loss", fontsize=12)
    plt.grid(True, linestyle=":", alpha=0.6)
    plt.legend(fontsize=12)
    plt.tight_layout()
    save_path="training_curve"+dataset_name+".png"
    # Save and display
    plt.savefig(save_path, dpi=300)
    plt.show()
    print(f"Plot saved to {save_path}")

In [ ]:
def train(
    model,
    device,
    ctx,
    scaler,
    config_data=None,
    batch_size=32,
    block_size=128,
    gradient_accumulation_steps=32,
    learning_rate=5e-4,
    min_lr=5e-5,
    warmup_steps=1000,
    max_iters=20000,
    eval_iters=100,
    eval_interval=500,
    dataset_dir="./",
    dataset_name="wiki",
    checkpoint_dir="./checkpoints",
    seed=42
):
    torch.manual_seed(seed)
    os.makedirs(checkpoint_dir, exist_ok=True)

    # Optimizer
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, betas=(0.9, 0.95), weight_decay=1e-1)

    # For plotting
    train_history = []
    val_history = []
    eval_steps = []

    # Helper: Cosine Learning Rate Schedule
    def get_lr(it):
        if it < warmup_steps:
            return learning_rate * it / warmup_steps
        if it > max_iters:
            return min_lr
        decay_ratio = (it - warmup_steps) / (max_iters - warmup_steps)
        coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
        return min_lr + coeff * (learning_rate - min_lr)

    # State Tracking & Initial Batch
    best_val_loss = float('inf')
    X, Y = get_batch('train', dataset_dir, dataset_name, block_size, batch_size, device)
    t0 = time.time()

    print("Starting training loop...")

    for iter_num in range(max_iters):
        # Update Learning Rate
        lr = get_lr(iter_num)
        for param_group in optimizer.param_groups:
            param_group['lr'] = lr


          # Periodic Validation & Checkpointing
        if iter_num % eval_interval == 0 or iter_num == max_iters - 1:
            losses = estimate_loss(
                model, eval_iters, dataset_dir, dataset_name,
                block_size, batch_size, device, ctx
            )
            train_loss = losses['train'].item() if hasattr(losses['train'], 'item') else losses['train']
            val_loss = losses['val'].item() if hasattr(losses['val'], 'item') else losses['val']

            # Record metrics
            eval_steps.append(iter_num)
            train_history.append(train_loss)
            val_history.append(val_loss)

            # Save checkpoint with history
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_path = os.path.join(checkpoint_dir, "best_model.pt")
                torch.save({
                    'model_state_dict': model.state_dict(),
                    'val_loss': val_loss,
                    'train_history': train_history,
                    'val_history': val_history,
                    'eval_steps': eval_steps
                }, best_path)

        # Forward & Backward Pass with Gradient Accumulation
        optimizer.zero_grad(set_to_none=True)

        for micro_step in range(gradient_accumulation_steps):
            with ctx:
                logits, loss = model(X, targets=Y) if hasattr(model, 'targets') else (model(X), None)
                if loss is None:
                    loss = torch.nn.functional.cross_entropy(logits.view(-1, logits.size(-1)), Y.view(-1))
                loss = loss / gradient_accumulation_steps

            X, Y = get_batch('train', dataset_dir, dataset_name, block_size, batch_size, device)
            scaler.scale(loss).backward()

        # Gradient Clipping & Optimizer Step
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        # Step Timer Logging
        t1 = time.time()
        dt = t1 - t0
        t0 = t1
        if iter_num % 10 == 0:
            print(f"iter {iter_num}: loss {loss.item() * gradient_accumulation_steps:.4f}, time {dt*1000:.2f}ms")
    plot_loss_curves(train_history, val_history, eval_interval=eval_interval,dataset_name=dataset_name)
    print("Training complete."
    return model

In [ ]:
@torch.no_grad()
def evaluate_test_set(model, dataset_dir, dataset_name, block_size, batch_size, eval_iters, device, ctx):
    """Calculates test loss and perplexity on the test split."""
    model.eval()
    losses = torch.zeros(eval_iters)

    for k in range(eval_iters):
        X_test, Y_test = get_batch('test', dataset_dir, dataset_name, block_size, batch_size, device)
        with ctx:
            logits, loss = model(X_test, targets=Y_test) if hasattr(model, 'targets') else (model(X_test), None)
            if loss is None:
                loss = torch.nn.functional.cross_entropy(logits.view(-1, logits.size(-1)), Y_test.view(-1))
        losses[k] = loss.item()

    test_loss = losses.mean().item()
    perplexity = math.exp(test_loss)

    print(f"Test Loss: {test_loss:.4f} | Perplexity: {perplexity:.2f}")
    return test_loss, perplexity

In [ ]:
if __name__ == "__main__":
    batch_size = 32
    block_size = 128
    gradient_accumulation_steps = 32
    #learning_rate = 5e-4
    learning_rate = 2e-4
    max_iters = 1500
    #max_iters = 7500
    #max_iters = 1500
    eval_iters = 100
    dataset_dir = work_dir
    dataset_name = "wiki"


    device, dtype, ctx, scaler = get_device_and_precision()
    config_data=get_config()
    gpt_model=init_model(config_data)
    gpt_model.to(device)
    trained_model = train(
        model=gpt_model,
        device=device,
        ctx=ctx,
        scaler=scaler,
        config_data=config_data,
        batch_size=batch_size,
        block_size=block_size,
        gradient_accumulation_steps=gradient_accumulation_steps,
        learning_rate=learning_rate,
        max_iters=max_iters,
        dataset_dir=dataset_dir,
        dataset_name=dataset_name
    )

    # Evaluate model on test set
    test_loss, perplexity = evaluate_test_set(
        model=trained_model,
        dataset_dir=dataset_dir,
        dataset_name=dataset_name,
        block_size=block_size,
        batch_size=batch_size,
        eval_iters=eval_iters,
        device=device,
        ctx=ctx
    )
    print("Dataset Name ",dataset_name," \n")
    print(f"\n[Final Evaluation] Test Loss: {test_loss:.4f} | Perplexity: {perplexity:.2f}")

    ### Switch to tinystories
    dataset_name = "tinystories"
    print("Dataset Name ",dataset_name," \n")
    device, dtype, ctx, scaler = get_device_and_precision()
    config_data=get_config()
    gpt_model=init_model(config_data)
    gpt_model.to(device)


    trained_model = train(
        model=gpt_model,
        device=device,
        ctx=ctx,
        scaler=scaler,
        config_data=config_data,
        batch_size=batch_size,
        block_size=block_size,
        gradient_accumulation_steps=gradient_accumulation_steps,
        learning_rate=learning_rate,
        max_iters=max_iters,
        dataset_dir=dataset_dir,
        dataset_name=dataset_name
    )

    # 2. Evaluate test loss & compute Perplexity
    test_loss, perplexity = evaluate_test_set(
        model=trained_model,
        dataset_dir=dataset_dir,
        dataset_name=dataset_name,
        block_size=block_size,
        batch_size=batch_size,
        eval_iters=eval_iters,
        device=device,
        ctx=ctx
    )
    print(f"\n[Final Evaluation] Test Loss: {test_loss:.4f} | Perplexity: {perplexity:.2f}")

    # 3. Answer Prompts / Story Generation
    test_prompts = [
        "Once upon a time, there was a tiny dog named",
        "Lily found a magic key in the garden.",
        "One sunny day, Tim went to the store to buy"
    ]

    print("\n--- Model Output Generations ---")
    for prompt in test_prompts:
        completion = generate(
            model=trained_model,
            tokenizer=tokenizer,
            prompt=prompt,
            max_new_tokens=80,
            temperature=0.7,
            device=device
        )
        print(f"\nPrompt: {prompt}\nGenerated Response:\n{completion}\n" + "-"*40)

**bold text** Train and test tiny stories